In [78]:
import os
import sys
import shutil
import zipfile
from geopy.geocoders import Nominatim

In [59]:
root = '/mnt/c/Users/user/Desktop/slc_cache/251229r'
geolocator = Nominatim(user_agent="slc_batch_labeller_semiauto_satgeolab")
available = os.listdir(root)

#filenames are assumed to be a flattened list of pairs (e.g. A1,A2,B1,B2,C1,C2,...)
filenames = 'ALPSRP067391560-L1.1, ALPSRP060681560-L1.1, ALPSRP067391570-L1.1, ALPSRP060681570-L1.1, ALPSRP067521420-L1.1, ALPSRP060811420-L1.1, ALPSRP068281250-L1.1, ALPSRP061571250-L1.1, ALPSRP068301420-L1.1, ALPSRP061591420-L1.1, ALPSRP068301430-L1.1, ALPSRP061591430-L1.1, ALPSRP068411540-L1.1, ALPSRP061701540-L1.1, ALPSRP068411600-L1.1, ALPSRP061701600-L1.1, ALPSRP068601390-L1.1, ALPSRP061891390-L1.1, ALPSRP069141540-L1.1, ALPSRP062431540-L1.1, ALPSRP069141560-L1.1, ALPSRP062431560-L1.1, ALPSRP069901400-L1.1, ALPSRP063191400-L1.1, ALPSRP069901410-L1.1, ALPSRP063191410-L1.1, ALPSRP070161530-L1.1, ALPSRP063451530-L1.1, ALPSRP123581400-L1.1, ALPSRP116871400-L1.1, ALPSRP123581410-L1.1, ALPSRP116871410-L1.1, ALPSRP070301320-L1.1, ALPSRP063591320-L1.1, ALPSRP070301310-L1.1, ALPSRP063591310-L1.1, ALPSRP071181420-L1.1, ALPSRP064471420-L1.1, ALPSRP071621550-L1.1, ALPSRP064911550-L1.1, ALPSRP071751420-L1.1, ALPSRP065041420-L1.1, ALPSRP071621570-L1.1, ALPSRP064911570-L1.1, ALPSRP071621560-L1.1, ALPSRP064911560-L1.1, ALPSRP072281560-L1.1, ALPSRP065571560-L1.1, ALPSRP125921430-L1.1, ALPSRP119211430-L1.1, ALPSRP072531420-L1.1, ALPSRP065821420-L1.1, ALPSRP072641600-L1.1, ALPSRP065931600-L1.1, ALPSRP072641610-L1.1, ALPSRP065931610-L1.1, ALPSRP233871330-L1.1, ALPSRP227161330-L1.1, ALPSRP128401430-L1.1, ALPSRP121691430-L1.1, ALPSRP235031430-L1.1, ALPSRP228321430-L1.1, ALPSRP185991400-L1.1, ALPSRP179281400-L1.1, ALPSRP185991490-L1.1, ALPSRP179281490-L1.1, ALPSRP185991410-L1.1, ALPSRP179281410-L1.1, ALPSRP187461570-L1.1, ALPSRP180751570-L1.1'
filenames = [el + '.zip' for el in filenames.split(', ') if (el+'.zip') in available]

for i in range(len(filenames)//2):
    p1 = zipfile.ZipFile(f'{root}/{filenames[i*2]}')
    p2 = zipfile.ZipFile(f'{root}/{filenames[i*2+1]}')
    with p1.open([el for el in p1.namelist() if 'workreport' in el][0]) as zipf:
        data = {el.split(b'=')[0]:el.split(b'=')[1].replace(b'\n',b'').replace(b'"',b'') for el in zipf}
        date1 = data[b'Img_SceneCenterDateTime'][:8].decode()
        lat1 = float(data[b'Brs_ImageSceneCenterLatitude'])
        lon1 = float(data[b'Brs_ImageSceneCenterLongitude'])
        id1 = data[b'Scs_SceneID'].decode()
    with p2.open([el for el in p2.namelist() if 'workreport' in el][0]) as zipf:
        data = {el.split(b'=')[0]:el.split(b'=')[1].replace(b'\n',b'').replace(b'"',b'') for el in zipf}
        date2 = data[b'Img_SceneCenterDateTime'][:8].decode()
        lat2 = float(data[b'Brs_ImageSceneCenterLatitude'])
        lon2 = float(data[b'Brs_ImageSceneCenterLongitude'])
        id2 = data[b'Scs_SceneID'].decode()

    llatlon = str((lat1+lat2)/2)+', '+str((lon1+lon2)/2)
    location = geolocator.reverse(llatlon)
    if (location is None): location = 'Offshore'
    else: location = location.raw['display_name'].replace(', ','-').replace(' ','_')
    location = ''.join([el for el in location if el.isalnum() or el in ['_','-']])

    fn_constructed = f'{id1}-{id2}-{date1}-{date2}'
    os.mkdir(f'{root}/{fn_constructed}')
    with open(f'{root}/{fn_constructed}/{location}.lbl', 'w', encoding='utf-8') as f: f.write('')
    os.rename(f'{root}/{filenames[i*2]}',f'{root}/{fn_constructed}/{filenames[i*2]}')
    os.rename(f'{root}/{filenames[i*2+1]}',f'{root}/{fn_constructed}/{filenames[i*2+1]}')
    
    print(fn_constructed)

ALPSRP072281560-ALPSRP065571560-20070603-20070418
ALPSRP125921430-ALPSRP119211430-20080605-20080420
ALPSRP072531420-ALPSRP065821420-20070605-20070420
ALPSRP072641600-ALPSRP065931600-20070606-20070421
ALPSRP072641610-ALPSRP065931610-20070606-20070421
ALPSRP233871330-ALPSRP227161330-20100615-20100430
ALPSRP128401430-ALPSRP121691430-20080622-20080507
ALPSRP235031430-ALPSRP228321430-20100623-20100508
ALPSRP185991400-ALPSRP179281400-20090722-20090606
ALPSRP185991490-ALPSRP179281490-20090722-20090606
ALPSRP185991410-ALPSRP179281410-20090722-20090606
ALPSRP187461570-ALPSRP180751570-20090801-20090616
